# Notebook 29G — exploratory V6-C retrain with source-balance candidate clips

**Runtime: T4 GPU.** Press **Run all**. This reuses the exact V6-C architecture, features, sampler, seed, augmentation, validation/test groups, and threshold-selection rule. The only training-data change is three 15-FPS landmark clips from two new original videos: Rowan Row (dumbbell row) and Nick Walker (sit-up). Inputs download automatically from pinned GitHub commit `58d9040`; no video, Drive mount, or manual upload is needed.

**Exploratory only:** the added clips have real visual first-pass annotations but no independent second review. The source package is deliberately marked `quarantined` and `training_allowed=false`. This notebook explicitly opts into those exact three clips for a diagnostic retrain; its checkpoint must not replace the deployed or frozen V6-C model. Inspect the returned per-class and creator metrics before deciding whether the data is worth independently reviewing and promoting.


In [ ]:
%pip install -q scikit-learn seaborn pandas==2.2.3


In [ ]:
from pathlib import Path
import hashlib
import json
import random
import shutil
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from google.colab import files
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

SEED = 42
WINDOW = 64
STRIDE = 32
BATCH_SIZE = 64
MAX_EPOCHS = 40
PATIENCE = 8
GROUP_TEMPERATURE = 0.5  # frozen from V6-A.1
AUGMENTATION_CONFIG = {
    "horizontal_flip_probability": 0.5,
    "rotation_probability": 0.7,
    "max_rotation_degrees": 7.0,
    "speed_probability": 0.5,
    "speed_range": [0.85, 1.15],
    "frame_drop_probability": 0.35,
    "frame_drop_fraction_range": [0.05, 0.12],
    "coordinate_noise_probability": 0.8,
    "coordinate_noise_std": 0.008,
}
RETIRED_CLASSES = {"plank"}
NEW_TRAIN_GROUPS = {"youtube:jTRwWrnVGuE", "youtube:6MYtjzEF6IA"}
UNREVIEWED_GROUPS = {
    "youtube:oLsonHQJOUc",
    "youtube:c4TomZjzobY",
    "youtube:W2Bm9dfQjbk",
}


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Experiment: exploratory exact V6-C plus three draft source-balance clips")
assert DEVICE.type == "cuda", "Select a T4 runtime before training."


## Download immutable evidence automatically

The seven original landmark archives, Notebook 13 weights, frozen V6 split,
V6-C baseline metrics/predictions, and the exact three-clip draft archive come
from pinned GitHub commit `58d9040`. Every downloaded input has a fixed
SHA-256 hash and every ZIP is checked before extraction. The three new clips
are consumed **only for this exploratory training run**.


In [ ]:
PINNED_COMMIT = "58d904049ba6fac03c4eadb741a55287464cc929"
REPOSITORY_RAW = (
    "https://raw.githubusercontent.com/bariscelikk1/cALorie/"
    f"{PINNED_COMMIT}/ml/artifacts"
)
LANDMARK_ARCHIVES = [
    ("v1/landmarks", "calorie_workoutfitness_landmarks_v1.zip"),
    ("v1/landmarks", "calorie_mmfit_landmarks_v1.zip"),
    ("v1/landmarks", "calorie_mmfit_unknown_v1.zip"),
    ("v2/landmarks", "calorie_youtube_mixed_landmarks_v3.zip"),
    ("v3/landmarks", "calorie_youtube_additional_landmarks_v1.zip"),
    ("v3/landmarks", "calorie_youtube_training_landmarks_v4.zip"),
    ("v5/landmarks", "calorie_workoutfitness_gate_hardneg_v1.zip"),
    ("source_balance_v1", "calorie_source_balance_v1_draft_clips.zip"),
]
EXPECTED_HASHES = {
    "calorie_source_balance_v1_draft_clips.zip": "721d69f30ec61b4c5d6a6d1a0a9782ec44b33c6511ac068458c86e0f9b671887",
    "calorie_workoutfitness_landmarks_v1.zip": "6dd4828e73f1ac1cd83a175de5bc1caf8465cbf61b72777467fba540af339aa9",
    "calorie_mmfit_landmarks_v1.zip": "2d3bff2c52842b276562e749fd7eaefe4925385efa7ff8c242c82ba1803413f1",
    "calorie_mmfit_unknown_v1.zip": "b0b113b3c5c7fc501b6b3ac4a7d2dade5fb4bccf3416d2eb85fa8b9197ae5596",
    "calorie_youtube_mixed_landmarks_v3.zip": "bd0426eda35654a95a8bac70b0357ba484e82bd29e6b3f71760fcc743f432cf4",
    "calorie_youtube_additional_landmarks_v1.zip": "0c60478086400fdfa6974afc6b091023b46e10783a71e40929aaa2b646d02743",
    "calorie_youtube_training_landmarks_v4.zip": "e0588c9a4a7d0257f710fec74b657e863cda5e0bc3285a640443863c9ede2b19",
    "calorie_workoutfitness_gate_hardneg_v1.zip": "b3aab2dbcefd709f1977c4d8d3f7d0326b50444ab44db4ba6b0cf64bbae0f4f6",
    "hard_positive_windows.csv": "5a4b73c04ee85659570e7931ea0e84b35c0abeb16fd3f08238e629f6eca091c5",
    "split_assignments.csv": "8c013695fe095ceccc4ff65722d2a26e5a309674f5b877b840b71ba73dc085d6",
    "baseline_metrics.json": "baee2ea099402ca25b0e2bc322809dfa7a0f8d1e9969204792cd5d60bfb96b2b",
    "baseline_test_predictions.csv": "b34b6cada90355682ef7aa225e3d18dad0960dd79ce32856ebb524356734326c",
}


def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def download(url, destination, expected=None):
    if destination.exists() and (expected is None or sha256(destination) == expected):
        print("Cached:", destination.name)
        return
    temporary = destination.with_suffix(destination.suffix + ".part")
    temporary.unlink(missing_ok=True)
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=300) as response, temporary.open("wb") as output:
        shutil.copyfileobj(response, output)
    if expected is not None:
        actual = sha256(temporary)
        assert actual == expected, f"SHA-256 mismatch for {destination.name}: {actual}"
    temporary.replace(destination)
    print(f"{destination.name}: {destination.stat().st_size / 1e6:.1f} MB")


zip_paths = []
for relative_root, name in LANDMARK_ARCHIVES:
    destination = Path("/content") / name
    download(
        f"{REPOSITORY_RAW}/{relative_root}/{name}", destination,
        EXPECTED_HASHES.get(name),
    )
    with zipfile.ZipFile(destination) as archive:
        assert archive.testzip() is None, f"Corrupt ZIP: {name}"
    zip_paths.append(destination)

HARD_POSITIVE_PATH = Path("/content/hard_positive_windows.csv")
download(
    f"{REPOSITORY_RAW}/v5/hard_positive_mining/hard_positive_windows.csv",
    HARD_POSITIVE_PATH, EXPECTED_HASHES["hard_positive_windows.csv"],
)
hard_positive_frame = pd.read_csv(HARD_POSITIVE_PATH)
assert hard_positive_frame.window_key.is_unique

V6_SPLIT_PATH = Path("/content/split_assignments.csv")
download(
    f"{REPOSITORY_RAW}/v6a1/model/split_assignments.csv",
    V6_SPLIT_PATH, EXPECTED_HASHES["split_assignments.csv"],
)
BASELINE_METRICS_PATH = Path("/content/baseline_metrics.json")
download(
    f"{REPOSITORY_RAW}/v6c/model/metrics.json",
    BASELINE_METRICS_PATH, EXPECTED_HASHES["baseline_metrics.json"],
)
BASELINE_PREDICTIONS_PATH = Path("/content/baseline_test_predictions.csv")
download(
    f"{REPOSITORY_RAW}/v6c/model/test_window_predictions.csv",
    BASELINE_PREDICTIONS_PATH, EXPECTED_HASHES["baseline_test_predictions.csv"],
)
baseline_metrics = json.loads(BASELINE_METRICS_PATH.read_text())
baseline_predictions = pd.read_csv(BASELINE_PREDICTIONS_PATH)

SOURCE_ROOT = Path("/content/v6c_source_balance_draft_sources")
if SOURCE_ROOT.exists():
    shutil.rmtree(SOURCE_ROOT)
SOURCE_ROOT.mkdir()
for index, archive_path in enumerate(zip_paths):
    destination = SOURCE_ROOT / f"source_{index}"
    destination.mkdir()
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(destination)

frames = []
for manifest_path in SOURCE_ROOT.rglob("manifest.csv"):
    frame = pd.read_csv(manifest_path).fillna("")
    frame["npz_path"] = frame["output_path"].map(
        lambda value: str(manifest_path.parent / value)
    )
    frames.append(frame)
assert frames, "No manifest.csv was found."
manifest = pd.concat(frames, ignore_index=True)
# The draft archive is quarantined by design. Opt into its exact three rows
# for this diagnostic only; never silently admit another unreviewed source.
draft_metadata = json.loads((SOURCE_ROOT / "source_7" / "run_metadata.json").read_text())
assert draft_metadata["training_allowed"] is False
assert draft_metadata["independent_second_review_complete"] is False
assert draft_metadata["clip_count"] == 3 and draft_metadata["group_count"] == 2
draft_rows = manifest[manifest.source_dataset == "source_balance_v1_first_pass"]
assert len(draft_rows) == 3
assert set(draft_rows.clip_id) == {"rowan_row_02_01", "rowan_row_03_01", "nick_sit_01_01"}
assert (draft_rows.status == "quarantined").all()
assert (draft_rows.review_status == "draft").all()
assert (~draft_rows.training_allowed.astype(str).str.lower().eq("true")).all()
assert set(draft_rows.canonical_class) == {"dumbbell_row", "sit_up"}
assert set(draft_rows.group_id) == NEW_TRAIN_GROUPS
candidate_mask = manifest.source_dataset == "source_balance_v1_first_pass"
assert not manifest.loc[~candidate_mask, "status"].eq("quarantined").any()
manifest = manifest[
    ((manifest.status == "ok") | (candidate_mask & manifest.status.eq("quarantined")))
    & manifest.npz_path.map(lambda value: Path(value).exists())
].copy()
if "group_id" not in manifest or manifest.group_id.eq("").any():
    manifest["group_id"] = manifest.apply(
        lambda row: f"{row['source_dataset']}:{row['clip_id']}", axis=1
    )
assert NEW_TRAIN_GROUPS <= set(manifest.group_id)
print("Valid clips:", len(manifest), "Groups:", manifest.group_id.nunique())
display(manifest.canonical_class.value_counts().sort_index().rename("clips"))


## Rebuild the exact old windows, then add two candidate train-only groups

All old groups reuse the frozen V6-A.1 assignment. The notebook asserts that
the complete old window set and all 1,971 V6-C test keys and labels remain
unchanged. Only the two new candidate groups are assigned to training. Rowan's
two clips are one group, so this experiment cannot establish broad creator
coverage or resolve all four source-confounded classes.


In [ ]:
def resample_sequence(sequence, source_fps, target_fps=15):
    if not source_fps or abs(float(source_fps) - target_fps) < 0.1:
        return sequence
    duration = (len(sequence) - 1) / float(source_fps)
    target_length = max(2, int(round(duration * target_fps)) + 1)
    old_x = np.linspace(0, duration, len(sequence))
    new_x = np.linspace(0, duration, target_length)
    flat = sequence.reshape(len(sequence), -1)
    sampled = np.stack([
        np.interp(new_x, old_x, flat[:, index])
        for index in range(flat.shape[1])
    ], axis=1)
    return sampled.reshape(target_length, 17, 4).astype(np.float32)


def interpolate_missing_pose_values(sequence):
    """Linearly repair sparse non-finite pose values along the time axis."""
    repaired = sequence.astype(np.float32, copy=True)
    missing = ~np.isfinite(repaired)
    missing_values = int(missing.sum())
    missing_frames = int(missing.any(axis=(1, 2)).sum())
    if not missing_values:
        return repaired, missing_values, missing_frames
    frame_axis = np.arange(len(repaired))
    for joint in range(repaired.shape[1]):
        for channel in range(repaired.shape[2]):
            values = repaired[:, joint, channel]
            valid = np.isfinite(values)
            assert valid.any(), (joint, channel)
            if not valid.all():
                repaired[:, joint, channel] = np.interp(
                    frame_axis, frame_axis[valid], values[valid]
                )
    assert np.isfinite(repaired).all()
    return repaired, missing_values, missing_frames


windows, original_labels, groups = [], [], []
clip_ids, sources, starts = [], [], []
pose_imputation_rows = []
for row in manifest.itertuples(index=False):
    with np.load(row.npz_path, allow_pickle=False) as item:
        sequence = item["landmarks"].astype(np.float32)
    if sequence.ndim != 3 or sequence.shape[1:] != (17, 4):
        print("Skipping unexpected shape:", row.npz_path, sequence.shape)
        continue
    sequence, missing_values, missing_frames = interpolate_missing_pose_values(sequence)
    if missing_values:
        pose_imputation_rows.append({
            "clip_id": row.clip_id,
            "group_id": row.group_id,
            "source_dataset": row.source_dataset,
            "missing_values_repaired": missing_values,
            "missing_frames_repaired": missing_frames,
            "repair_method": "per_joint_channel_linear_time_interpolation",
        })
    sequence = resample_sequence(sequence, getattr(row, "source_fps", 15) or 15)
    clip_starts = list(range(0, max(1, len(sequence) - WINDOW + 1), STRIDE))
    if len(sequence) > WINDOW and clip_starts[-1] != len(sequence) - WINDOW:
        clip_starts.append(len(sequence) - WINDOW)
    for start in clip_starts:
        window = sequence[start:start + WINDOW]
        if len(window) < WINDOW:
            window = np.concatenate([
                window, np.repeat(window[-1:], WINDOW - len(window), axis=0)
            ])
        windows.append(window)
        original_labels.append(row.canonical_class)
        groups.append(row.group_id)
        clip_ids.append(row.clip_id)
        sources.append(row.source_dataset)
        starts.append(start)

X = np.asarray(windows, dtype=np.float32)
assert np.isfinite(X).all(), "Non-finite pose values remain after temporal interpolation"
pose_imputation_audit = pd.DataFrame(pose_imputation_rows, columns=[
    "clip_id", "group_id", "source_dataset",
    "missing_values_repaired", "missing_frames_repaired", "repair_method",
])
print("Pose repair audit:", len(pose_imputation_audit), "clips,",
      int(pose_imputation_audit.missing_frames_repaired.sum()) if len(pose_imputation_audit) else 0,
      "frames")
original_labels = np.asarray(original_labels)
groups = np.asarray(groups)
clip_ids = np.asarray(clip_ids)
sources = np.asarray(sources)
starts = np.asarray(starts, dtype=np.int64)
window_keys = np.asarray([
    f"{source}|{group}|{clip}|{start}"
    for source, group, clip, start in zip(sources, groups, clip_ids, starts)
])

# Reuse the exact group assignment exported by V6-A.1.
frozen_split = pd.read_csv(V6_SPLIT_PATH)
assert frozen_split.groupby("group_id")["split"].nunique().max() == 1
group_to_split = (
    frozen_split[["group_id", "split"]]
    .drop_duplicates("group_id")
    .set_index("group_id")["split"]
    .to_dict()
)
old_evidence_mask = np.asarray([group in group_to_split for group in groups])
assigned_split = np.asarray([group_to_split.get(group, "") for group in groups])
assert NEW_TRAIN_GROUPS.isdisjoint(group_to_split), "Candidate group overlaps frozen split"
new_train_mask = np.isin(groups, list(NEW_TRAIN_GROUPS))
assigned_split[new_train_mask] = "train"
missing_groups = set(groups[assigned_split == ""])
assert missing_groups <= UNREVIEWED_GROUPS, (
    "Unexpected unresolved groups: " + str(sorted(missing_groups))
)
assert set(groups[new_train_mask]) == NEW_TRAIN_GROUPS
train_idx = np.flatnonzero(assigned_split == "train")
val_idx = np.flatnonzero(assigned_split == "validation")
test_idx = np.flatnonzero(assigned_split == "test")

labels = np.where(np.isin(original_labels, list(RETIRED_CLASSES)), "unknown", original_labels)
ALL_CLASSES = sorted(np.unique(labels).tolist())
SUPPORTED_CLASSES = [name for name in ALL_CLASSES if name != "unknown"]
assert len(SUPPORTED_CLASSES) == 14, SUPPORTED_CLASSES
all_to_id = {name: index for index, name in enumerate(ALL_CLASSES)}
supported_to_id = {name: index for index, name in enumerate(SUPPORTED_CLASSES)}
y_all = np.asarray([all_to_id[name] for name in labels], dtype=np.int64)
y_gate = (labels != "unknown").astype(np.int64)
y_exercise = np.asarray([supported_to_id.get(name, -1) for name in labels], dtype=np.int64)

assert not (set(groups[train_idx]) & set(groups[val_idx]))
assert not (set(groups[train_idx]) & set(groups[test_idx]))
assert not (set(groups[val_idx]) & set(groups[test_idx]))
assert not (set(groups[train_idx]) & UNREVIEWED_GROUPS)
assert set(window_keys[old_evidence_mask]) == set(frozen_split.window_key), (
    "Reviewed window evidence changed since V6-A.1."
)
assert set(window_keys[test_idx]) == set(baseline_predictions.window_key)
assert len(test_idx) == 1971, len(test_idx)
baseline_truth_by_key = dict(zip(baseline_predictions.window_key, baseline_predictions.true_class))
assert all(baseline_truth_by_key[key] == truth for key, truth in zip(window_keys[test_idx], labels[test_idx])), "Frozen test labels changed relative to V6-C."
assert not set(groups[val_idx]) & NEW_TRAIN_GROUPS
assert not set(groups[test_idx]) & NEW_TRAIN_GROUPS
print("Excluded unresolved groups:", sorted(missing_groups))
print("New train-only groups:", sorted(NEW_TRAIN_GROUPS))

print("Windows:", X.shape)
print("Frozen groups:", len(group_to_split))
print("Active supported classes:", SUPPORTED_CLASSES)
print("Retired plank windows mapped to unknown:", int((original_labels == "plank").sum()))
for name, indices in [("train", train_idx), ("validation", val_idx), ("test", test_idx)]:
    print(name, "windows=", len(indices), "groups=", len(np.unique(groups[indices])))
    display(pd.Series(labels[indices]).value_counts().sort_index().rename("windows"))


new_indices = np.flatnonzero(new_train_mask)
data_delta_audit = pd.DataFrame({
    "canonical_class": labels[new_indices],
    "group_id": groups[new_indices],
    "window_key": window_keys[new_indices],
}).groupby(["canonical_class", "group_id"]).agg(
    windows=("window_key", "count")
).reset_index()
class_creator_support_matrix = pd.crosstab(
    pd.Series(labels[train_idx], name="canonical_class"),
    pd.Series(groups[train_idx], name="group_id"),
)
print("Added training windows:", len(new_indices))
display(data_delta_audit)
display(class_creator_support_matrix)


## Tempered hierarchical sampling and hard-positive weighting

Sampling first assigns equal probability to each class. Inside a class, each
source group receives probability proportional to `window_count ** 0.5`. This is
the midpoint between failed V6-A equal-group sampling (`** 0`) and raw window
sampling (`** 1`): long videos are restrained without making a one-window group
as influential as a large group. Notebook 13 hard-positive modifiers are
normalized inside the same class/group, so they change which windows are seen
without changing that group's total probability.

For the gate, “class” means supported or unknown. For the exercise model it
means one of the 14 active movements. `WeightedRandomSampler` then draws the
requested class -> group -> window distribution.


In [ ]:
hard_weight_by_key = dict(zip(
    hard_positive_frame.window_key,
    hard_positive_frame.recommended_sample_weight,
))
hard_positive_weights = np.asarray([
    float(hard_weight_by_key.get(key, 1.0)) for key in window_keys
], dtype=np.float64)
# Retired plank is now unknown and cannot receive a supported hard-positive boost.
hard_positive_weights[labels == "unknown"] = 1.0
matched_training_keys = (
    set(window_keys[train_idx][labels[train_idx] != "unknown"])
    & set(hard_weight_by_key)
)
assert not (set(window_keys[val_idx]) & set(hard_weight_by_key)), (
    "Notebook 13 weights unexpectedly intersect validation."
)
assert not (set(window_keys[test_idx]) & set(hard_weight_by_key)), (
    "Notebook 13 weights unexpectedly intersect test."
)
print("Notebook 13 keys:", len(hard_weight_by_key))
print("Matched active training keys:", len(matched_training_keys))
print("Upweighted active training windows:", int((hard_positive_weights[train_idx] > 1).sum()))


def hierarchical_window_weights(indices, hierarchy_labels, extra_weights=None):
    table = pd.DataFrame({
        "index": indices,
        "hierarchy_label": hierarchy_labels[indices],
        "group": groups[indices],
    })
    class_count = table["hierarchy_label"].nunique()
    group_sizes = table.groupby(
        ["hierarchy_label", "group"]
    )["index"].transform("count").astype(float)
    table["group_mass"] = group_sizes.pow(GROUP_TEMPERATURE)
    group_table = table[["hierarchy_label", "group"]].copy()
    group_table["group_mass"] = table["group_mass"]
    class_group_mass = (
        group_table.drop_duplicates(["hierarchy_label", "group"])
        .groupby("hierarchy_label")["group_mass"].sum()
    )
    table["extra_weight"] = (
        1.0 if extra_weights is None else extra_weights[table["index"].to_numpy()]
    )
    table["pair_extra_total"] = table.groupby(
        ["hierarchy_label", "group"]
    )["extra_weight"].transform("sum")
    weights = []
    for row in table.itertuples(index=False):
        weight = (
            1.0 / class_count
            * row.group_mass / class_group_mass.loc[row.hierarchy_label]
            * row.extra_weight / row.pair_extra_total
        )
        weights.append(weight)
    weights = np.asarray(weights, dtype=np.float64)
    return weights / weights.mean()


# Use strings here so the audit table remains human-readable.
gate_hierarchy = np.where(y_gate == 1, "supported", "unknown")
gate_train_weights = hierarchical_window_weights(
    train_idx, gate_hierarchy, hard_positive_weights
)
exercise_train_idx = train_idx[y_exercise[train_idx] >= 0]
exercise_val_idx = val_idx[y_exercise[val_idx] >= 0]
exercise_test_idx = test_idx[y_exercise[test_idx] >= 0]
exercise_train_weights = hierarchical_window_weights(
    exercise_train_idx, labels
)

sampler_audit = pd.DataFrame({
    "window_key": window_keys[train_idx],
    "canonical_class": labels[train_idx],
    "original_class": original_labels[train_idx],
    "group_id": groups[train_idx],
    "gate_target": gate_hierarchy[train_idx],
    "hard_positive_weight": hard_positive_weights[train_idx],
    "gate_sampler_weight": gate_train_weights,
})
exercise_weight_map = dict(zip(
    window_keys[exercise_train_idx], exercise_train_weights
))
sampler_audit["exercise_sampler_weight"] = sampler_audit.window_key.map(
    exercise_weight_map
).fillna(0.0)

display(sampler_audit.groupby("gate_target").agg(
    windows=("window_key", "count"),
    groups=("group_id", "nunique"),
    total_sampler_weight=("gate_sampler_weight", "sum"),
).round(3))
display(sampler_audit[sampler_audit.canonical_class != "unknown"].groupby(
    "canonical_class"
).agg(
    windows=("window_key", "count"),
    groups=("group_id", "nunique"),
    total_exercise_weight=("exercise_sampler_weight", "sum"),
).round(3))


In [ ]:
LEFT_RIGHT_PAIRS = [
    (1, 2), (3, 4), (5, 6), (7, 8),
    (9, 10), (11, 12), (13, 14), (15, 16),
]


def augment_pose_window(sample, force_all=False):
    sample = sample.copy()
    config = AUGMENTATION_CONFIG

    if force_all or np.random.random() < config["horizontal_flip_probability"]:
        sample[:, :, 0] *= -1.0
        for left, right in LEFT_RIGHT_PAIRS:
            sample[:, [left, right], :] = sample[:, [right, left], :]

    if force_all or np.random.random() < config["rotation_probability"]:
        degrees = np.random.uniform(
            -config["max_rotation_degrees"], config["max_rotation_degrees"]
        )
        radians = np.deg2rad(degrees)
        cosine, sine = np.cos(radians), np.sin(radians)
        x = sample[:, :, 0].copy()
        y = sample[:, :, 1].copy()
        sample[:, :, 0] = cosine * x - sine * y
        sample[:, :, 1] = sine * x + cosine * y

    if force_all or np.random.random() < config["speed_probability"]:
        speed = np.random.uniform(*config["speed_range"])
        center = (WINDOW - 1) / 2.0
        positions = (np.arange(WINDOW) - center) * speed + center
        flat = sample.reshape(WINDOW, -1)
        flat = np.stack([
            np.interp(positions, np.arange(WINDOW), flat[:, column])
            for column in range(flat.shape[1])
        ], axis=1)
        sample = flat.reshape(WINDOW, 17, 4).astype(np.float32)

    if force_all or np.random.random() < config["frame_drop_probability"]:
        fraction = np.random.uniform(*config["frame_drop_fraction_range"])
        count = max(1, int(round(WINDOW * fraction)))
        dropped = np.random.choice(np.arange(1, WINDOW), size=count, replace=False)
        sample[dropped] = sample[dropped - 1]

    if force_all or np.random.random() < config["coordinate_noise_probability"]:
        sample[:, :, :2] += np.random.normal(
            0, config["coordinate_noise_std"], sample[:, :, :2].shape
        ).astype(np.float32)
    return sample.astype(np.float32)



BONE_EDGES = [
    (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),
    (5, 11), (6, 12), (11, 12),
    (11, 13), (13, 15), (12, 14), (14, 16),
]
ANGLE_TRIPLETS = [
    (5, 7, 9), (6, 8, 10),       # elbows
    (7, 5, 11), (8, 6, 12),      # shoulders
    (5, 11, 13), (6, 12, 14),    # hips
    (11, 13, 15), (12, 14, 16),  # knees
]
FEATURE_CONFIG = {
    "coordinates_xy": 34,
    "velocity_xy": 34,
    "bone_vectors_xy": 24,
    "joint_angles": 8,
}
FEATURE_DIM = sum(FEATURE_CONFIG.values())


def pose_to_features(sample):
    """Convert one [time, 17, 4] normalized pose window to [time, 100]."""
    xy = sample[:, :, :2].astype(np.float32)

    velocity = np.zeros_like(xy)
    velocity[1:] = xy[1:] - xy[:-1]

    bones = np.stack([
        xy[:, child] - xy[:, parent] for parent, child in BONE_EDGES
    ], axis=1)

    angles = []
    for first, center, last in ANGLE_TRIPLETS:
        first_vector = xy[:, first] - xy[:, center]
        last_vector = xy[:, last] - xy[:, center]
        denominator = (
            np.linalg.norm(first_vector, axis=1)
            * np.linalg.norm(last_vector, axis=1)
        )
        cosine = np.sum(first_vector * last_vector, axis=1) / np.maximum(
            denominator, 1e-6
        )
        angles.append(np.arccos(np.clip(cosine, -1.0, 1.0)) / np.pi)
    angles = np.stack(angles, axis=1).astype(np.float32)

    features = np.concatenate([
        xy.reshape(WINDOW, -1),
        velocity.reshape(WINDOW, -1),
        bones.reshape(WINDOW, -1),
        angles,
    ], axis=1).astype(np.float32)
    assert features.shape == (WINDOW, FEATURE_DIM), features.shape
    assert np.isfinite(features).all()
    return features


class PoseWindowDataset(Dataset):
    def __init__(self, indices, targets, training=False):
        self.indices = np.asarray(indices)
        self.targets = targets
        self.training = training

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        actual = self.indices[item]
        sample = X[actual].copy()
        if self.training:
            sample = augment_pose_window(sample)
        features = pose_to_features(sample).T
        return torch.from_numpy(features), torch.tensor(
            self.targets[actual], dtype=torch.long
        )


def make_loaders(targets, train_indices, val_indices, test_indices, train_weights, seed):
    generator = torch.Generator().manual_seed(seed)
    sampler = WeightedRandomSampler(
        torch.as_tensor(train_weights, dtype=torch.double),
        len(train_weights), replacement=True, generator=generator,
    )
    return (
        DataLoader(
            PoseWindowDataset(train_indices, targets, True),
            batch_size=BATCH_SIZE, sampler=sampler,
        ),
        DataLoader(
            PoseWindowDataset(val_indices, targets),
            batch_size=BATCH_SIZE, shuffle=False,
        ),
        DataLoader(
            PoseWindowDataset(test_indices, targets),
            batch_size=BATCH_SIZE, shuffle=False,
        ),
    )


gate_loaders = make_loaders(
    y_gate, train_idx, val_idx, test_idx, gate_train_weights, SEED
)
exercise_loaders = make_loaders(
    y_exercise, exercise_train_idx, exercise_val_idx, exercise_test_idx,
    exercise_train_weights, SEED + 1,
)
print("Feature blocks:", FEATURE_CONFIG)
print("Input features per frame:", FEATURE_DIM)
print("Gate split sizes:", [len(loader.dataset) for loader in gate_loaders])
print("Exercise split sizes:", [len(loader.dataset) for loader in exercise_loaders])


## Audit the augmentation before training

The preview applies every transformation once to a real training window. Check
that limbs remain connected, left/right swapping is anatomical, and motion is
perturbed without becoming a different exercise. The image is included in the
report for reproducibility.


In [ ]:
SKELETON_EDGES = [
    (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),
    (5, 11), (6, 12), (11, 12),
    (11, 13), (13, 15), (12, 14), (14, 16),
]
preview_candidates = train_idx[labels[train_idx] == "squat"]
preview_index = int(preview_candidates[0] if len(preview_candidates) else train_idx[0])
original_preview = X[preview_index].copy()
random_state = np.random.get_state()
np.random.seed(SEED + 1000)
augmented_preview = augment_pose_window(original_preview, force_all=True)
np.random.set_state(random_state)

fig, axes = plt.subplots(2, 6, figsize=(15, 6))
frame_indices = np.linspace(0, WINDOW - 1, 6, dtype=int)
for row, (title, sequence) in enumerate([
    ("original", original_preview), ("augmented", augmented_preview)
]):
    for axis, frame_index in zip(axes[row], frame_indices):
        xy = sequence[frame_index, :, :2]
        for left, right in SKELETON_EDGES:
            axis.plot(
                [xy[left, 0], xy[right, 0]],
                [-xy[left, 1], -xy[right, 1]],
                color="tab:blue", linewidth=1.4,
            )
        axis.scatter(xy[:, 0], -xy[:, 1], s=12, color="tab:red")
        axis.set_aspect("equal")
        axis.axis("off")
        axis.set_title(f"{title} t={frame_index}")
fig.suptitle(f"Notebook 24 frozen augmentation audit | {labels[preview_index]} | {window_keys[preview_index]}")
fig.tight_layout()
AUGMENTATION_PREVIEW_PATH = Path("/content/v6b_augmentation_preview.png")
fig.savefig(AUGMENTATION_PREVIEW_PATH, dpi=170, bbox_inches="tight")
plt.show()
print("Preview:", AUGMENTATION_PREVIEW_PATH)


## Frozen V6-C model and feature representation

The 100-channel coordinates, velocity, bone vectors, and joint angles; TCN
blocks; optimizer; scheduler; early stopping; and random seeds are copied from
Notebook 16. No architecture or loss change is introduced here.


In [ ]:
class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dilation, dropout=0.2):
        super().__init__()
        padding = dilation
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, 3, padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_channels), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(out_channels, out_channels, 3, padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_channels), nn.ReLU(), nn.Dropout(dropout),
        )
        self.residual = (
            nn.Conv1d(in_channels, out_channels, 1)
            if in_channels != out_channels else nn.Identity()
        )

    def forward(self, x):
        return torch.relu(self.net(x) + self.residual(x))


class SmallTCN(nn.Module):
    def __init__(self, input_features, num_classes):
        super().__init__()
        channels = [64, 96, 128, 128]
        blocks, current = [], input_features
        for level, output in enumerate(channels):
            blocks.append(TemporalBlock(current, output, dilation=2 ** level))
            current = output
        self.features = nn.Sequential(*blocks)
        self.classifier = nn.Linear(current, num_classes)

    def forward(self, x):
        return self.classifier(self.features(x).mean(dim=-1))


In [ ]:
def predict_model(model, loader):
    model.eval()
    truths, probabilities = [], []
    with torch.no_grad():
        for features, targets in loader:
            logits = model(features.to(DEVICE))
            truths.extend(targets.numpy())
            probabilities.extend(logits.softmax(1).cpu().numpy())
    probabilities = np.asarray(probabilities)
    return np.asarray(truths), probabilities.argmax(1), probabilities


def train_model(name, num_classes, loaders, seed):
    seed_everything(seed)
    train_loader, val_loader, _ = loaders
    model = SmallTCN(FEATURE_DIM, num_classes).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=3, factor=0.5
    )
    best_path = Path(f"/content/best_v6c_source_balance_draft_{name}.pt")
    best_f1, stale, history = -1.0, 0, []

    def run_epoch(loader, training):
        model.train(training)
        losses, truths, predictions = [], [], []
        context = torch.enable_grad() if training else torch.no_grad()
        with context:
            for features, targets in loader:
                features, targets = features.to(DEVICE), targets.to(DEVICE)
                if training:
                    optimizer.zero_grad(set_to_none=True)
                logits = model(features)
                loss = criterion(logits, targets)
                if training:
                    loss.backward()
                    optimizer.step()
                losses.append(loss.item() * len(targets))
                truths.extend(targets.cpu().numpy())
                predictions.extend(logits.argmax(1).cpu().numpy())
        return (
            sum(losses) / len(truths),
            f1_score(truths, predictions, average="macro", zero_division=0),
        )

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss, train_f1 = run_epoch(train_loader, True)
        val_loss, val_f1 = run_epoch(val_loader, False)
        scheduler.step(val_f1)
        history.append({
            "model": name, "epoch": epoch,
            "train_loss": train_loss, "train_macro_f1": train_f1,
            "val_loss": val_loss, "val_macro_f1": val_f1,
        })
        print(
            f"{name} {epoch:02d} train loss={train_loss:.4f} F1={train_f1:.3f} "
            f"| val loss={val_loss:.4f} F1={val_f1:.3f}"
        )
        if val_f1 > best_f1 + 1e-4:
            best_f1, stale = val_f1, 0
            torch.save(model.state_dict(), best_path)
        else:
            stale += 1
            if stale >= PATIENCE:
                print(name, "early stopping")
                break

    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    return model, history, best_f1


gate_model, gate_history, gate_best_f1 = train_model(
    "gate", 2, gate_loaders, SEED
)
exercise_model, exercise_history, exercise_best_f1 = train_model(
    "exercise", len(SUPPORTED_CLASSES), exercise_loaders, SEED + 1
)
print("Best validation F1 — gate:", round(gate_best_f1, 4))
print("Best validation F1 — exercise:", round(exercise_best_f1, 4))


## Frozen validation-only threshold selection

The threshold sweep uses the unchanged group-separated validation set and the
same selection rule as V6-C. New training groups never enter validation or
test.


In [ ]:
_, _, gate_val_probability = predict_model(gate_model, gate_loaders[1])


def full_probabilities(indices):
    features = np.stack([
        pose_to_features(X[index]) for index in indices
    ]).transpose(0, 2, 1)
    gate_parts, exercise_parts = [], []
    gate_model.eval()
    exercise_model.eval()
    with torch.no_grad():
        for start in range(0, len(features), 256):
            batch = torch.from_numpy(features[start:start + 256]).to(DEVICE)
            gate_parts.append(gate_model(batch).softmax(1).cpu().numpy())
            exercise_parts.append(exercise_model(batch).softmax(1).cpu().numpy())
    return np.concatenate(gate_parts), np.concatenate(exercise_parts)


_, exercise_val_probabilities = full_probabilities(val_idx)
exercise_val_predictions = exercise_val_probabilities.argmax(1)
threshold_rows = []
for threshold in np.linspace(0.05, 0.95, 37):
    end_predictions = np.asarray([
        SUPPORTED_CLASSES[class_id] if probability >= threshold else "unknown"
        for probability, class_id in zip(
            gate_val_probability[:, 1], exercise_val_predictions
        )
    ])
    supported_mask = labels[val_idx] != "unknown"
    unknown_mask = ~supported_mask
    threshold_rows.append({
        "threshold": float(threshold),
        "validation_macro_f1": float(f1_score(
            labels[val_idx], end_predictions, labels=ALL_CLASSES,
            average="macro", zero_division=0,
        )),
        "supported_accuracy": float(
            (end_predictions[supported_mask] == labels[val_idx][supported_mask]).mean()
        ),
        "unknown_rejection": float(
            (end_predictions[unknown_mask] == "unknown").mean()
        ),
    })
threshold_frame = pd.DataFrame(threshold_rows)
best_threshold_row = threshold_frame.sort_values(
    ["validation_macro_f1", "supported_accuracy"], ascending=False
).iloc[0]
GATE_THRESHOLD = float(best_threshold_row.threshold)
print("Interim validation threshold:", GATE_THRESHOLD)
display(threshold_frame.sort_values("validation_macro_f1", ascending=False).head(10))


## Unchanged internal test and auditable export

Evaluation uses the exact 1,971 V6-C test windows. The report records aggregate,
per-class, and creator-level deltas, data-support counts, source hashes,
sampler weights, predictions, checkpoints, plots, and split assignments.
Because candidate annotations remain draft, the checkpoint is exploratory.


In [ ]:
gate_test_prob, exercise_test_prob = full_probabilities(test_idx)
gate_test_pred = gate_test_prob.argmax(1)
exercise_test_pred = exercise_test_prob.argmax(1)
end_to_end_pred = np.asarray([
    SUPPORTED_CLASSES[class_id] if probability >= GATE_THRESHOLD else "unknown"
    for probability, class_id in zip(gate_test_prob[:, 1], exercise_test_pred)
])

supported_test_mask = labels[test_idx] != "unknown"
classifier_truth = labels[test_idx][supported_test_mask]
classifier_pred = np.asarray([
    SUPPORTED_CLASSES[index] for index in exercise_test_pred[supported_test_mask]
])
metrics = {
    "experiment": "v6c_exact_plus_source_balance_draft_exploratory",
    "candidate_annotation_stage": "first_pass_only",
    "independent_second_review_complete": False,
    "checkpoint_promotion_allowed": False,
    "active_supported_class_count": len(SUPPORTED_CLASSES),
    "retired_classes_mapped_to_unknown": sorted(RETIRED_CLASSES),
    "training_archives": [path.name for path in zip_paths],
    "pinned_source_commit": PINNED_COMMIT,
    "new_train_groups": sorted(NEW_TRAIN_GROUPS),
    "added_training_windows": int(new_train_mask.sum()),
    "unchanged_test_windows": int(len(test_idx)),
    "only_changed_block": "three draft clips from two new groups added to training for diagnostic use only",
    "seed": SEED,
    "window_frames": WINDOW,
    "stride_frames": STRIDE,
    "augmentation": AUGMENTATION_CONFIG,
    "feature_representation": FEATURE_CONFIG,
    "input_features": FEATURE_DIM,
    "sampler": {
        "group_temperature": GROUP_TEMPERATURE,
        "gate": "binary_class_then_sqrt_group_mass_then_window_x_hard_positive_weight",
        "exercise": "canonical_class_then_sqrt_group_mass_then_window",
        "matched_hard_positive_training_windows": len(matched_training_keys),
    },
    "gate_threshold": GATE_THRESHOLD,
    "threshold_selection": "interim group-separated validation; not final calibration",
    "gate": {
        "accuracy": float(accuracy_score(y_gate[test_idx], gate_test_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_gate[test_idx], gate_test_pred)),
        "macro_f1": float(f1_score(
            y_gate[test_idx], gate_test_pred, average="macro", zero_division=0
        )),
    },
    "exercise_classifier_oracle": {
        "accuracy": float(accuracy_score(classifier_truth, classifier_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(classifier_truth, classifier_pred)),
        "macro_f1": float(f1_score(
            classifier_truth, classifier_pred, labels=SUPPORTED_CLASSES,
            average="macro", zero_division=0,
        )),
    },
    "end_to_end": {
        "accuracy": float(accuracy_score(labels[test_idx], end_to_end_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(labels[test_idx], end_to_end_pred)),
        "macro_f1": float(f1_score(
            labels[test_idx], end_to_end_pred, labels=ALL_CLASSES,
            average="macro", zero_division=0,
        )),
    },
    "classes": ALL_CLASSES,
    "supported_classes": SUPPORTED_CLASSES,
}
baseline_primary = {
    "gate_macro_f1": baseline_metrics["gate"]["macro_f1"],
    "exercise_oracle_macro_f1": baseline_metrics["exercise_classifier_oracle"]["macro_f1"],
    "end_to_end_macro_f1": baseline_metrics["end_to_end"]["macro_f1"],
    "end_to_end_accuracy": baseline_metrics["end_to_end"]["accuracy"],
}
current_primary = {
    "gate_macro_f1": metrics["gate"]["macro_f1"],
    "exercise_oracle_macro_f1": metrics["exercise_classifier_oracle"]["macro_f1"],
    "end_to_end_macro_f1": metrics["end_to_end"]["macro_f1"],
    "end_to_end_accuracy": metrics["end_to_end"]["accuracy"],
}
metrics["v6c_baseline"] = baseline_primary
metrics["delta_vs_v6c"] = {
    key: current_primary[key] - baseline_primary[key] for key in current_primary
}
print(json.dumps(metrics, indent=2))

REPORT_ROOT = Path("/content/calorie_v6c_source_balance_draft_retrain_report")
if REPORT_ROOT.exists():
    shutil.rmtree(REPORT_ROOT)
REPORT_ROOT.mkdir()
shutil.copy2(AUGMENTATION_PREVIEW_PATH, REPORT_ROOT / "augmentation_preview.png")

pd.DataFrame(gate_history + exercise_history).to_csv(
    REPORT_ROOT / "training_history.csv", index=False
)
threshold_frame.to_csv(REPORT_ROOT / "gate_threshold_sweep.csv", index=False)
sampler_audit.to_csv(REPORT_ROOT / "sampler_audit.csv", index=False)
(REPORT_ROOT / "metrics.json").write_text(json.dumps(metrics, indent=2) + "\n")
(REPORT_ROOT / "baseline_v6c_metrics.json").write_text(
    json.dumps(baseline_metrics, indent=2) + "\n"
)
data_delta_audit.to_csv(REPORT_ROOT / "data_delta_audit.csv", index=False)
pose_imputation_audit.to_csv(REPORT_ROOT / "pose_imputation_audit.csv", index=False)
class_creator_support_matrix.to_csv(REPORT_ROOT / "class_creator_support_matrix.csv")
source_hashes = {
    path.name: sha256(path) for path in zip_paths
}
source_hashes.update({
    HARD_POSITIVE_PATH.name: sha256(HARD_POSITIVE_PATH),
    V6_SPLIT_PATH.name: sha256(V6_SPLIT_PATH),
    BASELINE_METRICS_PATH.name: sha256(BASELINE_METRICS_PATH),
    BASELINE_PREDICTIONS_PATH.name: sha256(BASELINE_PREDICTIONS_PATH),
})
(REPORT_ROOT / "source_hashes.json").write_text(
    json.dumps(source_hashes, indent=2) + "\n"
)

split_rows = []
for split_name, indices in [("train", train_idx), ("validation", val_idx), ("test", test_idx)]:
    split_rows.extend({
        "window_key": window_keys[index],
        "group_id": groups[index],
        "clip_id": clip_ids[index],
        "source_dataset": sources[index],
        "original_class": original_labels[index],
        "active_class": labels[index],
        "split": split_name,
    } for index in indices)
pd.DataFrame(split_rows).to_csv(REPORT_ROOT / "split_assignments.csv", index=False)

reports = {
    "gate_classification_report.json": classification_report(
        y_gate[test_idx], gate_test_pred, labels=[0, 1],
        target_names=["unknown", "supported"], output_dict=True, zero_division=0,
    ),
    "exercise_classification_report.json": classification_report(
        classifier_truth, classifier_pred, labels=SUPPORTED_CLASSES,
        output_dict=True, zero_division=0,
    ),
    "end_to_end_classification_report.json": classification_report(
        labels[test_idx], end_to_end_pred, labels=ALL_CLASSES,
        output_dict=True, zero_division=0,
    ),
}
for filename, report in reports.items():
    (REPORT_ROOT / filename).write_text(json.dumps(report, indent=2))

plots = [
    ("gate", y_gate[test_idx], gate_test_pred, [0, 1], ["unknown", "supported"]),
    ("exercise", classifier_truth, classifier_pred, SUPPORTED_CLASSES, SUPPORTED_CLASSES),
    ("end_to_end", labels[test_idx], end_to_end_pred, ALL_CLASSES, ALL_CLASSES),
]
for name, truth, prediction, matrix_labels, tick_labels in plots:
    matrix = confusion_matrix(truth, prediction, labels=matrix_labels)
    plt.figure(figsize=(max(6, len(tick_labels) * 0.75), max(5, len(tick_labels) * 0.65)))
    sns.heatmap(
        matrix, annot=True, fmt="d", cmap="Blues",
        xticklabels=tick_labels, yticklabels=tick_labels,
    )
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(name.replace("_", " ").title() + " test confusion matrix")
    plt.tight_layout()
    plt.savefig(REPORT_ROOT / f"{name}_confusion_matrix.png", dpi=180)
    plt.show()

history_frame = pd.DataFrame(gate_history + exercise_history)
for name, group in history_frame.groupby("model"):
    group.plot(
        x="epoch", y=["train_macro_f1", "val_macro_f1"], figsize=(8, 4),
        grid=True, title=name,
    )
    plt.tight_layout()
    plt.savefig(REPORT_ROOT / f"{name}_learning_curve.png", dpi=180)
    plt.show()

baseline_prediction_by_key = dict(zip(
    baseline_predictions.window_key, baseline_predictions.predicted_class
))
baseline_end_to_end_pred = np.asarray([
    baseline_prediction_by_key[key] for key in window_keys[test_idx]
])
creator_rows = []
for group_id in sorted(set(groups[test_idx])):
    mask = groups[test_idx] == group_id
    truth = labels[test_idx][mask]
    prediction = end_to_end_pred[mask]
    baseline_prediction = baseline_end_to_end_pred[mask]
    present = sorted(set(truth))
    current_f1 = float(f1_score(
        truth, prediction, labels=present, average="macro", zero_division=0
    ))
    baseline_f1 = float(f1_score(
        truth, baseline_prediction, labels=present, average="macro", zero_division=0
    ))
    creator_rows.append({
        "group_id": group_id,
        "windows": int(mask.sum()),
        "present_classes": "|".join(present),
        "baseline_accuracy": float(accuracy_score(truth, baseline_prediction)),
        "current_accuracy": float(accuracy_score(truth, prediction)),
        "accuracy_delta": float(
            accuracy_score(truth, prediction)
            - accuracy_score(truth, baseline_prediction)
        ),
        "baseline_present_class_macro_f1": baseline_f1,
        "current_present_class_macro_f1": current_f1,
        "present_class_macro_f1_delta": current_f1 - baseline_f1,
        "unknown_recall": float(
            (prediction[truth == "unknown"] == "unknown").mean()
        ) if (truth == "unknown").any() else None,
    })
pd.DataFrame(creator_rows).to_csv(
    REPORT_ROOT / "test_creator_breakdown.csv", index=False
)

pd.DataFrame({
    "window_key": window_keys[test_idx],
    "clip_id": clip_ids[test_idx],
    "source_dataset": sources[test_idx],
    "true_class": labels[test_idx],
    "gate_supported_probability": gate_test_prob[:, 1],
    "exercise_prediction": [SUPPORTED_CLASSES[index] for index in exercise_test_pred],
    "exercise_confidence": exercise_test_prob.max(axis=1),
    "predicted_class": end_to_end_pred,
}).to_csv(REPORT_ROOT / "test_window_predictions.csv", index=False)

torch.save({
    "gate_state_dict": gate_model.state_dict(),
    "exercise_state_dict": exercise_model.state_dict(),
    "classes": ALL_CLASSES,
    "supported_classes": SUPPORTED_CLASSES,
    "retired_classes": sorted(RETIRED_CLASSES),
    "gate_threshold": GATE_THRESHOLD,
    "window_frames": WINDOW,
    "input_features": FEATURE_DIM,
    "feature_representation": FEATURE_CONFIG,
    "architecture": "two_stage_small_tcn_v6c_kinematic_features",
    "pinned_source_commit": PINNED_COMMIT,
    "new_train_groups": sorted(NEW_TRAIN_GROUPS),
    "experiment": metrics["experiment"],
    "candidate_annotation_stage": "first_pass_only",
    "checkpoint_promotion_allowed": False,
}, REPORT_ROOT / "two_stage_tcn_v6c_source_balance_draft.pt")

checksum_lines = []
for report_path in sorted(path for path in REPORT_ROOT.rglob("*") if path.is_file()):
    checksum_lines.append(
        f"{sha256(report_path)}  {report_path.relative_to(REPORT_ROOT)}"
    )
(REPORT_ROOT / "SHA256SUMS.txt").write_text("\n".join(checksum_lines) + "\n")

zip_path = shutil.make_archive(
    "/content/calorie_v6c_source_balance_draft_retrain_report", "zip", root_dir=REPORT_ROOT
)
print("Report ZIP:", zip_path)
print("ZIP size MB:", round(Path(zip_path).stat().st_size / 1024**2, 2))
files.download(zip_path)


## Return the exploratory report

The final cell automatically downloads
`calorie_v6c_source_balance_draft_retrain_report.zip`. Send that ZIP back for
hash verification and baseline comparison. Do not promote its checkpoint based
on this run; the added clips still need independent visual review.
